In [3]:
import glob, os, re
import numpy as np
import pandas as pd

DATA_DIR = "../labeled"

REMOVE = ["Respirate", "Leave_Home", "Enter_Home", "O Relax", "OF Meal_Preparation",
          "CLOSE Enter_Home", "CLOSE Leave_Home", "OPEN Enter_Home", "OPEN Leave_Home"]

def read_home(path):
    rows = []
    with open(path, errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            p = [x.strip() for x in line.split(",")] if "," in line else line.split()
            if len(p) < 4:
                continue
            ts = pd.to_datetime(p[0] + " " + p[1], errors="coerce")
            if pd.isna(ts):
                continue
            state = p[-1].lower() if p[-1].lower() in ("begin", "end") else None
            label = " ".join(p[4:-1]) if state and len(p) >= 6 else None
            rows.append((ts, label, state))
    return pd.DataFrame(rows, columns=["timestamp", "label", "state"])

def daily_pct(df):
    df = df[df["state"].notna() & df["label"].notna()]
    df = df[~df["label"].isin(REMOVE)].sort_values("timestamp")
    segs = []
    for lab, g in df.groupby("label"):
        start = None
        for ts, st in zip(g["timestamp"], g["state"]):
            if st == "begin":
                start = ts
            elif start is not None:
                segs.append((lab, start, ts))
                start = None
    rows = []
    for act, s, e in segs:
        while s.normalize() < e.normalize():
            nxt = s.normalize() + pd.Timedelta(days=1)
            rows.append((act, s.normalize(), (nxt - s).total_seconds() / 60))
            s = nxt
        rows.append((act, s.normalize(), (e - s).total_seconds() / 60))
    if not rows:
        return pd.DataFrame()
    o = pd.DataFrame(rows, columns=["activity", "date", "minutes"])
    daily = o.pivot_table(index="date", columns="activity", values="minutes",
                          aggfunc="sum", fill_value=0)
    return daily / 1440 * 100

homes = {}
for f in sorted(glob.glob(os.path.join(DATA_DIR, "*"))):
    name = os.path.basename(f).split(".")[0]
    if re.fullmatch(r"hh1[0-3]\d", name):
        pct = daily_pct(read_home(f))
        if len(pct):
            homes[name] = pct

print("homes loaded:", len(homes), "(paper: 30)")
print("total days:", sum(len(d) for d in homes.values()), "(paper: 2245)")
ALL_ACTS = sorted(set().union(*[set(d.columns) for d in homes.values()]))
print("activities:", len(ALL_ACTS), "(paper: up to 41)")


KeyboardInterrupt: 

In [2]:
with open("../labeled/hh101.csv", errors="ignore") as f:
    for i, line in enumerate(f):
        print(repr(line))
        if i == 3:
            break

'2012-07-20,10:38:54.512364,OutsideDoor,ON,Step_Out="begin"\n'
'2012-07-20,10:38:59.541365,OutsideDoor,OFF\n'
'2012-07-20,10:39:02.429671,OutsideDoor,ON\n'
'2012-07-20,10:39:04.120443,OutsideDoor,OFF\n'
